# CARDIODIAG Complete Research Notebook

This notebook is an executable, leakage-controlled research workflow. It runs only on datasets discovered from the release directory or documented Kaggle mounts. It reports internal validation, calibration, selective prediction, conformal coverage, and the corrected StableXAIHDRC feature-selection workflow without claiming external validity or clinical utility.

## 1. Environment and dependencies
The release pins compatible dependency ranges in `requirements-kaggle.txt` and `pyproject.toml`. This cell checks the runtime before any experiment.

In [ ]:
import importlib.util
import platform
from pathlib import Path

REQUIRED = ["numpy", "pandas", "scipy", "sklearn", "xgboost", "lightgbm", "catboost", "matplotlib"]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
print("Python:", platform.python_version())
print("Missing packages:", missing)
if missing:
    raise ImportError("Install the documented release dependencies before running the notebook: " + ", ".join(missing))


## 2. Configuration and reproducibility
All stochastic operations use the fixed seed below. Outer folds are held out for evaluation; inner folds select model hyperparameters.

In [ ]:
SEED = 42
CONFIG = {
    "seed": SEED,
    "outer_splits": 3,
    "inner_splits": 2,
    "primary_metric": "roc_auc",
    "classification_threshold": 0.5,
    "calibration_fraction": 0.25,
    "conformal_alpha": 0.1,
}
CONFIG


## 3. Dataset discovery, schema verification, and provenance
The release bundles only the Mendeley dataset. Heart5 and the Kaggle 70k dataset are supported when manually attached, but are never silently downloaded or substituted.

In [ ]:
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and Path("/kaggle/input").exists():
    candidates = list(Path("/kaggle/input").glob("**/src"))
    if candidates:
        PROJECT_ROOT = candidates[0].parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from cardiodiag.data import load_dataset, drop_identifier_columns, summarize_dataset
from cardiodiag.evaluation import evaluate_predictions
from cardiodiag.preprocessing import TrainingOnlyCleaner, make_tabular_preprocessor
from cardiodiag.provenance import collect_provenance
from cardiodiag.experiments import build_corrected_pipeline
from cardiodiag.reliability import calibration_summary, selective_risk_curve, uncertainty_evidence, nogueira_stability
from cardiodiag.uncertainty import SplitConformalClassifier

def find_dataset(filename):
    candidates = [PROJECT_ROOT / "data" / filename, PROJECT_ROOT / filename]
    if Path("/kaggle/input").exists():
        candidates += list(Path("/kaggle/input").glob(f"**/{filename}"))
    return next((p for p in candidates if p.exists()), None)

DATASETS = {
    "mendeley": (find_dataset("MendeleyCardiovascular_Disease_Dataset.csv"), "target"),
    "heart5": (find_dataset("heart5.csv"), "HeartDisease"),
    "kaggle70k": (find_dataset("cardio_train.csv"), "cardio"),
}
AVAILABLE = {name: pair for name, pair in DATASETS.items() if pair[0] is not None}
if not AVAILABLE:
    raise FileNotFoundError("Attach a permitted dataset according to KAGGLE_SETUP.md before running.")
DATASET_SUMMARIES = {}
PROVENANCE = {}
for name, (path, target) in AVAILABLE.items():
    DATASET_SUMMARIES[name] = summarize_dataset(path, target).__dict__
    PROVENANCE[name] = collect_provenance(path, target, "complete_research_notebook", CONFIG)
print("Project root:", PROJECT_ROOT)
print("Available datasets:", list(AVAILABLE))
print(json.dumps(DATASET_SUMMARIES, indent=2, default=str))


## 4. Data-quality checks
These checks validate target separation, binary labels, identifiers, schema metadata, duplicate counts, and finite numeric values before modeling.

In [ ]:
for name, (path, target) in AVAILABLE.items():
    X, y = load_dataset(path, target)
    assert target not in X.columns
    assert set(pd.unique(y)) == {0, 1}, (name, y.unique())
    X_model, dropped = drop_identifier_columns(X)
    assert target not in X_model.columns
    numeric = X_model.select_dtypes(include=["number"])
    assert np.isfinite(numeric.to_numpy(dtype=float)).all(), name
    print({"dataset": name, "rows": len(X), "dropped_identifiers": dropped, "categorical": list(X_model.attrs.get("categorical_columns", [])), "numeric": list(X_model.attrs.get("numeric_columns", [])), "duplicates": DATASET_SUMMARIES[name]["duplicate_rows"]})


## 5. Fold-local preprocessing and nested model selection
The pipeline fits cleaning, imputation, encoding, scaling, and the model inside each training fold. Grid search uses only inner training folds; outer predictions are untouched until evaluation.

In [ ]:
def pipeline_for(X, model):
    categorical = list(X.attrs.get("categorical_columns", []))
    numeric = list(X.attrs.get("numeric_columns", []))
    return Pipeline([
        ("cleaner", TrainingOnlyCleaner(categorical_columns=tuple(categorical))),
        ("preprocessor", make_tabular_preprocessor(numeric, categorical)),
        ("model", model),
    ])

def metric_record(y_true, probabilities):
    p = np.asarray(probabilities, dtype=float)
    labels = (p >= CONFIG["classification_threshold"]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, labels, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
        "accuracy": float(accuracy_score(y_true, labels)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, labels)),
        "precision": float(precision_score(y_true, labels, zero_division=0)),
        "recall": float(recall_score(y_true, labels, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if tn + fp else float("nan"),
        "f1": float(f1_score(y_true, labels, zero_division=0)),
        "brier": float(brier_score_loss(y_true, p)),
        "log_loss": float(log_loss(y_true, np.column_stack([1-p, p]), labels=[0, 1])),
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    }

def nested_oof(path, target):
    X, y = load_dataset(path, target)
    X, dropped = drop_identifier_columns(X)
    outer = StratifiedKFold(n_splits=CONFIG["outer_splits"], shuffle=True, random_state=SEED)
    specs = {
        "logistic": (LogisticRegression(max_iter=1500, class_weight="balanced", random_state=SEED), {"model__C": [0.1, 1.0, 10.0]}),
        "random_forest": (RandomForestClassifier(n_estimators=120, n_jobs=1, class_weight="balanced", random_state=SEED), {"model__max_depth": [None, 16]}),
        "hist_gradient_boosting": (HistGradientBoostingClassifier(random_state=SEED), {"model__max_leaf_nodes": [15, 31], "model__max_iter": [100]}),
    }
    outputs = {"dataset": str(path), "target": target, "dropped_identifiers": dropped, "configuration": CONFIG, "models": {}}
    for name, (model, grid) in specs.items():
        predictions = np.full(len(y), np.nan)
        folds = []
        for fold, (train_idx, test_idx) in enumerate(outer.split(X, y)):
            search = GridSearchCV(pipeline_for(X.iloc[train_idx], model), grid, scoring=CONFIG["primary_metric"], cv=CONFIG["inner_splits"], n_jobs=1, refit=True)
            search.fit(X.iloc[train_idx], y.iloc[train_idx])
            predictions[test_idx] = search.predict_proba(X.iloc[test_idx])[:, 1]
            folds.append({"fold": fold, "train_rows": len(train_idx), "test_rows": len(test_idx), "best_params": search.best_params_, "metrics": metric_record(y.iloc[test_idx].to_numpy(), predictions[test_idx])})
        assert np.isfinite(predictions).all()
        outputs["models"][name] = {"pooled": metric_record(y.to_numpy(), predictions), "folds": folds, "oof_probabilities": predictions.tolist()}
    return outputs


## 6. Nested-validation results
Each available dataset is evaluated independently. The stored OOF probabilities support downstream calibration and reliability analysis without refitting on outer-test labels.

In [ ]:
NESTED_RESULTS = {}
for name, (path, target) in AVAILABLE.items():
    started = time.perf_counter()
    NESTED_RESULTS[name] = nested_oof(path, target)
    NESTED_RESULTS[name]["runtime_seconds"] = time.perf_counter() - started
    print(name, {model: values["pooled"]["roc_auc"] for model, values in NESTED_RESULTS[name]["models"].items()})


## 7. OOF model comparison and fixed ensemble
The ensemble averages the three model probabilities with fixed equal weights. No outer-test labels select members or weights.

In [ ]:
for name, result in NESTED_RESULTS.items():
    y = load_dataset(AVAILABLE[name][0], AVAILABLE[name][1])[1].to_numpy()
    probability_arrays = [np.asarray(result["models"][model]["oof_probabilities"]) for model in result["models"]]
    ensemble = np.mean(probability_arrays, axis=0)
    result["fixed_soft_voting"] = metric_record(y, ensemble)
    print(name, "fixed_soft_voting", result["fixed_soft_voting"]["roc_auc"])


## 8. Calibration with a separate calibration partition
The base model is fitted on the training partition only. A logistic calibrator is fitted on a disjoint calibration partition and evaluated on a third evaluation partition.

In [ ]:
CALIBRATION_RESULTS = {}
for name, (path, target) in AVAILABLE.items():
    X, y = load_dataset(path, target); X, _ = drop_identifier_columns(X)
    train_idx, remainder = train_test_split(np.arange(len(y)), test_size=0.4, stratify=y, random_state=SEED)
    cal_idx, eval_idx = train_test_split(remainder, test_size=0.5, stratify=y.iloc[remainder], random_state=SEED)
    base = pipeline_for(X.iloc[train_idx], HistGradientBoostingClassifier(random_state=SEED, max_iter=100))
    base.fit(X.iloc[train_idx], y.iloc[train_idx])
    p_cal = np.clip(base.predict_proba(X.iloc[cal_idx])[:, 1], 1e-6, 1-1e-6)
    p_eval = np.clip(base.predict_proba(X.iloc[eval_idx])[:, 1], 1e-6, 1-1e-6)
    calibrator = LogisticRegression(random_state=SEED)
    calibrator.fit(np.log(p_cal / (1 - p_cal)).reshape(-1, 1), y.iloc[cal_idx])
    p_calibrated = calibrator.predict_proba(np.log(p_eval / (1 - p_eval)).reshape(-1, 1))[:, 1]
    CALIBRATION_RESULTS[name] = {"train_rows": len(train_idx), "calibration_rows": len(cal_idx), "evaluation_rows": len(eval_idx), "uncalibrated": metric_record(y.iloc[eval_idx].to_numpy(), p_eval), "calibrated": metric_record(y.iloc[eval_idx].to_numpy(), p_calibrated), "calibration_summary": calibration_summary(y.iloc[eval_idx].to_numpy(), p_calibrated)}
    print(name, CALIBRATION_RESULTS[name]["calibrated"]["brier"], CALIBRATION_RESULTS[name]["calibrated"]["log_loss"])


## 9. Reliability and selective prediction
ECE, confidence-error association, risk-coverage behavior, and full-coverage risk are calculated on the held-out evaluation partition.

In [ ]:
RELIABILITY_RESULTS = {}
for name, result in CALIBRATION_RESULTS.items():
    X, y = load_dataset(AVAILABLE[name][0], AVAILABLE[name][1]); X, _ = drop_identifier_columns(X)
    _, remainder = train_test_split(np.arange(len(y)), test_size=0.4, stratify=y, random_state=SEED)
    _, eval_idx = train_test_split(remainder, test_size=0.5, stratify=y.iloc[remainder], random_state=SEED)
    p = np.asarray(result["calibrated"]["confusion_matrix"]) if False else None
    # Reuse the calibrated metric inputs through a deterministic refit of the same partitions.
    train_idx, remainder = train_test_split(np.arange(len(y)), test_size=0.4, stratify=y, random_state=SEED)
    cal_idx, eval_idx = train_test_split(remainder, test_size=0.5, stratify=y.iloc[remainder], random_state=SEED)
    base = pipeline_for(X.iloc[train_idx], HistGradientBoostingClassifier(random_state=SEED, max_iter=100)).fit(X.iloc[train_idx], y.iloc[train_idx])
    p_cal = np.clip(base.predict_proba(X.iloc[cal_idx])[:,1], 1e-6, 1-1e-6); p_eval = np.clip(base.predict_proba(X.iloc[eval_idx])[:,1], 1e-6, 1-1e-6)
    calibrator = LogisticRegression(random_state=SEED).fit(np.log(p_cal/(1-p_cal)).reshape(-1,1), y.iloc[cal_idx])
    p = calibrator.predict_proba(np.log(p_eval/(1-p_eval)).reshape(-1,1))[:,1]
    y_eval = y.iloc[eval_idx].to_numpy()
    selective = selective_risk_curve(y_eval, p)
    evidence = uncertainty_evidence(y_eval, p)
    RELIABILITY_RESULTS[name] = {"calibration": result["calibration_summary"], "selective_risk_curve": selective, "uncertainty_evidence": evidence, "full_coverage_risk": float(np.mean((p >= 0.5) != y_eval))}
    print(name, "full_coverage_risk", RELIABILITY_RESULTS[name]["full_coverage_risk"])


## 10. Split-conformal prediction
The estimator is fitted on training data, calibration scores are computed on the separate calibration partition, and coverage is measured only on evaluation observations.

In [ ]:
CONFORMAL_RESULTS = {}
for name, (path, target) in AVAILABLE.items():
    X, y = load_dataset(path, target); X, _ = drop_identifier_columns(X)
    train_idx, remainder = train_test_split(np.arange(len(y)), test_size=0.4, stratify=y, random_state=SEED)
    cal_idx, eval_idx = train_test_split(remainder, test_size=0.5, stratify=y.iloc[remainder], random_state=SEED)
    estimator = pipeline_for(X.iloc[train_idx], HistGradientBoostingClassifier(random_state=SEED, max_iter=100))
    conformal = SplitConformalClassifier(estimator, alpha=CONFIG["conformal_alpha"]).fit(X.iloc[train_idx], y.iloc[train_idx], X.iloc[cal_idx], y.iloc[cal_idx])
    summary = conformal.summarize(X.iloc[eval_idx], y.iloc[eval_idx])
    CONFORMAL_RESULTS[name] = {"train_rows": len(train_idx), "calibration_rows": len(cal_idx), "evaluation_rows": len(eval_idx), **summary}
    print(name, CONFORMAL_RESULTS[name])


## 11. Corrected CARDIODIAG selector workflow
The corrected selector splits original observations before bootstrap sampling. This cell runs a fold-local corrected logistic pipeline and records selected-feature counts without using outer-test observations for fitting.

In [ ]:
SELECTOR_RESULTS = {}
for name, (path, target) in AVAILABLE.items():
    X, y = load_dataset(path, target); X, _ = drop_identifier_columns(X)
    train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEED)
    selector_pipeline = build_corrected_pipeline(X.iloc[train_idx], "logistic", seed=SEED, use_stable_selector=True)
    selector_pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
    p = selector_pipeline.predict_proba(X.iloc[test_idx])[:, 1]
    selector = selector_pipeline.named_steps.get("stable_xaih_drc")
    selected_count = int(np.sum(selector.get_support())) if selector is not None else None
    SELECTOR_RESULTS[name] = {"train_rows": len(train_idx), "test_rows": len(test_idx), "selected_feature_count": selected_count, "test_metrics": metric_record(y.iloc[test_idx].to_numpy(), p), "selector_protocol": "post_boundary_correction_v1"}
    print(name, SELECTOR_RESULTS[name])


## 12. Feature-selection stability
Repeated deterministic selector fits on training-only resamples produce a binary selection matrix. Nogueira stability is computed from that matrix; the test partition remains untouched.

In [ ]:
STABILITY_RESULTS = {}
for name, (path, target) in AVAILABLE.items():
    X, y = load_dataset(path, target); X, _ = drop_identifier_columns(X)
    train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEED)
    supports = []
    for repeat in range(3):
        sub_idx = train_idx[np.random.default_rng(SEED + repeat).choice(len(train_idx), size=len(train_idx), replace=True)]
        pipe = build_corrected_pipeline(X.iloc[sub_idx], "logistic", seed=SEED + repeat, use_stable_selector=True).fit(X.iloc[sub_idx], y.iloc[sub_idx])
        selector = pipe.named_steps["stable_xaih_drc"]
        supports.append(selector.get_support().astype(int))
    matrix = np.asarray(supports)
    STABILITY_RESULTS[name] = {"selection_matrix_shape": list(matrix.shape), "nogueira_stability": float(nogueira_stability(matrix)), "test_rows_excluded_from_fit": len(test_idx)}
    print(name, STABILITY_RESULTS[name])


## 13. Statistical summaries and intervals
Bootstrap intervals below are descriptive resampling intervals over observed OOF predictions, not independent-patient uncertainty intervals.

In [ ]:
def descriptive_bootstrap_auc(y, p, seed=SEED, repetitions=200):
    rng = np.random.default_rng(seed); values = []
    for _ in range(repetitions):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        values.append(float(roc_auc_score(y[idx], p[idx])))
    return {"lower": float(np.quantile(values, 0.025)), "upper": float(np.quantile(values, 0.975)), "repetitions": len(values), "unit": "observed rows; descriptive only"}

for name, result in NESTED_RESULTS.items():
    y = load_dataset(AVAILABLE[name][0], AVAILABLE[name][1])[1].to_numpy()
    for model, values in result["models"].items():
        values["descriptive_auc_interval"] = descriptive_bootstrap_auc(y, np.asarray(values["oof_probabilities"]))
print("Descriptive intervals added to nested results.")


## 14. Visual diagnostics
ROC, precision-recall, calibration, and risk-coverage figures are generated from held-out predictions and saved under `results/notebook_outputs/`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

FIGURE_DIR = PROJECT_ROOT / "results" / "notebook_outputs"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
for name, result in NESTED_RESULTS.items():
    y = load_dataset(AVAILABLE[name][0], AVAILABLE[name][1])[1].to_numpy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for model, values in result["models"].items():
        p = np.asarray(values["oof_probabilities"])
        RocCurveDisplay.from_predictions(y, p, ax=axes[0], name=model)
        PrecisionRecallDisplay.from_predictions(y, p, ax=axes[1], name=model)
        frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
        axes[2].plot(mean_pred, frac_pos, marker="o", label=model)
    axes[0].set_title("ROC"); axes[1].set_title("Precision-recall"); axes[2].plot([0,1],[0,1],"k--"); axes[2].set_title("Calibration"); axes[2].legend()
    fig.tight_layout(); fig.savefig(FIGURE_DIR / f"{name}_diagnostics.png", dpi=120); plt.close(fig)
print("Saved figures:", sorted(p.name for p in FIGURE_DIR.glob("*.png")))


## 15. Cross-dataset interpretation
Datasets are summarized separately. No pooled performance estimate or cross-dataset external-validation claim is made because schemas, sampling, target semantics, and licensing differ.

In [ ]:
CROSS_DATASET_SUMMARY = {name: {"rows": DATASET_SUMMARIES[name]["rows"], "target": target, "protocol": "independent internal validation only"} for name, (_, target) in AVAILABLE.items()}
CROSS_DATASET_SUMMARY


## 16. Save metrics, provenance, configurations, and limitations
The output is a new notebook artifact and never overwrites historical experiment JSON files.

In [ ]:
OUTPUT = {
    "configuration": CONFIG,
    "datasets": DATASET_SUMMARIES,
    "provenance": PROVENANCE,
    "nested_validation": NESTED_RESULTS,
    "calibration": CALIBRATION_RESULTS,
    "reliability": RELIABILITY_RESULTS,
    "conformal": CONFORMAL_RESULTS,
    "corrected_selector": SELECTOR_RESULTS,
    "feature_selection_stability": STABILITY_RESULTS,
    "cross_dataset_summary": CROSS_DATASET_SUMMARY,
    "limitations": [
        "Results are internal validation within each available dataset.",
        "Descriptive bootstrap intervals resample observed rows and are not independent-patient uncertainty.",
        "Excluded datasets require manual attachment and are not silently substituted.",
        "No clinical utility, deployment readiness, or universal generalization claim is supported.",
    ],
}
output_path = PROJECT_ROOT / "results" / "notebook_outputs" / "complete_research_metrics.json"
output_path.write_text(json.dumps(OUTPUT, indent=2, default=str), encoding="utf-8")
print("Saved", output_path, "bytes", output_path.stat().st_size)


## 17. Reproduction and execution audit
Restart the kernel before running all cells. Use only `requirements-kaggle.txt` or `uv sync`, attach permitted datasets according to `KAGGLE_SETUP.md`, and verify the saved JSON plus figures. The corrected selector protocol is explicitly recorded as `post_boundary_correction_v1`.